## 🔍 Stop Loss Logic Verification Summary

### ✅ Correct Implementation (Both cells now fixed)

**Key components of proper stop loss logic:**

1. **Trading Costs Applied:**
   - Entry: `entry_price = price * (1 + trading_cost_pct)` 
   - Exit: `exit_price = price * (1 - trading_cost_pct)`

2. **Stop Loss Priority (checked FIRST):**
   ```python
   if price <= stop_loss_price:  # Priority 1: Stop loss
       exit and calculate return
   elif prob <= sell_threshold:  # Priority 2: Sell signal
       exit and calculate return
   ```

3. **Percentage-Based Returns:**
   - ✅ `balance *= (exit_price / entry_price)` - Correct
   - ❌ `balance += (sell_price - buy_price)` - WRONG (was in old code)

4. **Stop Loss Calculation:**
   - `stop_loss_price = entry_price * (1 - stop_loss_pct)`
   - For 2% stop loss: exits when price drops 2% below entry price

### 📊 Both Cells Are Now Consistent:
- **Cell 9**: Simple strategy execution with detailed trade history
- **Cell 10**: Threshold optimization with stop loss (grid search)

## 5-Minute Trading Strategy Backtest

Load models and test them with backtesting based on balance and risk management techniques

In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import yfinance as yf
import talib as tal
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Try to import TensorFlow/Keras for neural network models
try:
    import tensorflow as tf
    from tensorflow import keras
    KERAS_AVAILABLE = True
    print("✅ TensorFlow/Keras available")
except ImportError:
    KERAS_AVAILABLE = False
    print("⚠️ TensorFlow/Keras not available - only sklearn models supported")

print("✅ Libraries imported successfully")

✅ TensorFlow/Keras available
✅ Libraries imported successfully


In [2]:
# ============================================================================
# LOAD ALL 5m MODELS FROM RESULTS FOLDER
# ============================================================================
import os
import json
import pickle
import pandas as pd

# Path to the results root folder
RESULTS_ROOT = r"C:\Users\aram\OneDrive\Bureau\ding\testing a strategy\python_classes\results"

def load_all_5m_models():
    """Load all models from the 5m results folder."""
    models_5m = {}

    # Find the 5m results folder
    results_5m_path = None
    for folder_name in os.listdir(RESULTS_ROOT):
        if folder_name.startswith('ETH_USD_5m_') and os.path.isdir(os.path.join(RESULTS_ROOT, folder_name)):
            results_5m_path = os.path.join(RESULTS_ROOT, folder_name)
            break

    if results_5m_path is None:
        print("❌ No 5m results folder found!")
        return models_5m

    print(f"📂 Found 5m results folder: {os.path.basename(results_5m_path)}")

    # Load config and results
    config_path = os.path.join(results_5m_path, "config.json")
    results_path = os.path.join(results_5m_path, "results_summary.json")
    registry_path = os.path.join(results_5m_path, "models_registry.json")
    features_path = os.path.join(results_5m_path, "features.json")

    if not os.path.exists(config_path) or not os.path.exists(results_path):
        print("❌ Missing config or results files!")
        return models_5m

    # Load metadata
    config = json.load(open(config_path))
    results = json.load(open(results_path))

    models_registry = None
    if os.path.exists(registry_path):
        try:
            models_registry = json.load(open(registry_path))
        except:
            models_registry = None

    features_config = None
    if os.path.exists(features_path):
        try:
            features_config = json.load(open(features_path))
        except:
            features_config = None

    # Get all models from ranking
    ranking = results.get('ranking', [])
    all_models_info = results.get('all_models', {})

    print(f"🔍 Found {len(ranking)} models in 5m results")
    print(f"🏆 Best model: {results.get('best_model', 'N/A')}")

    # Load each model
    models_dir = os.path.join(results_5m_path, "models")
    loaded_count = 0

    for model_info in ranking:
        model_name = model_info['model']

        # Find model file
        model_path = None
        model_type = None

        # Check models directory first
        if os.path.exists(models_dir):
            for ext in ['.keras', '.pkl']:
                candidate_path = os.path.join(models_dir, f"{model_name}{ext}")
                if os.path.exists(candidate_path):
                    model_path = candidate_path
                    model_type = 'keras' if ext == '.keras' else 'pkl'
                    break

        # Check root directory if not found
        if model_path is None:
            for ext in ['.keras', '.pkl']:
                candidate_path = os.path.join(results_5m_path, f"{model_name}{ext}")
                if os.path.exists(candidate_path):
                    model_path = candidate_path
                    model_type = 'keras' if ext == '.keras' else 'pkl'
                    break

        if model_path is None:
            print(f"⚠️ Model file not found for: {model_name}")
            continue

        # Load the model
        try:
            if model_type == 'keras' and KERAS_AVAILABLE:
                model = keras.models.load_model(model_path)
            elif model_type in ['pkl', 'sklearn']:
                with open(model_path, 'rb') as f:
                    model = pickle.load(f)
            else:
                print(f"⚠️ Unsupported model type {model_type} for: {model_name}")
                continue

            # Get additional info
            model_class = None
            if models_registry:
                model_entry = models_registry.get('models', {}).get(model_name, {})
                model_class = model_entry.get('model_class')

            # Store model info
            models_5m[model_name] = {
                'model': model,
                'model_type': model_type,
                'model_class': model_class,
                'model_path': model_path,
                'feature_set': model_info.get('feature_set', 'X'),
                'train_acc': model_info.get('train_acc', 0),
                'forward_acc': model_info.get('forward_acc', 0),
                'rank': model_info.get('rank', 999),
                'config': config,
                'features_config': features_config
            }

            loaded_count += 1
            print(f"✅ Loaded {model_name} ({model_type}) - Rank: {model_info.get('rank', 'N/A')}, Acc: {model_info.get('forward_acc', 0):.4f}")

        except Exception as e:
            print(f"❌ Failed to load {model_name}: {e}")
            continue

    print(f"\n📊 Successfully loaded {loaded_count} out of {len(ranking)} 5m models")
    return models_5m

# ============================================================================
# EXECUTE LOADING
# ============================================================================
print("=" * 70)
print("🚀 LOADING ALL 5m MODELS")
print("=" * 70)

MODELS_5m = load_all_5m_models()

if len(MODELS_5m) == 0:
    print("❌ No 5m models were loaded!")
else:
    print(f"\n📈 Available 5m models: {list(MODELS_5m.keys())}")

    # Show summary
    df_5m_models = pd.DataFrame([
        {
            'Model': name,
            'Type': info['model_type'],
            'Class': info['model_class'] or 'N/A',
            'Rank': info['rank'],
            'Train_Acc': f"{info['train_acc']:.4f}",
            'Forward_Acc': f"{info['forward_acc']:.4f}",
            'Feature_Set': info['feature_set']
        }
        for name, info in MODELS_5m.items()
    ]).sort_values('Rank')

    print("\n🏆 5m MODELS SUMMARY (sorted by rank):")
    display(df_5m_models)

print("=" * 70)

🚀 LOADING ALL 5m MODELS
📂 Found 5m results folder: ETH_USD_5m_20260116_113846
🔍 Found 30 models in 5m results
🏆 Best model: AdaBoost_Full
✅ Loaded AdaBoost_Full (pkl) - Rank: 1, Acc: 0.5212
✅ Loaded AdaBoost_X (pkl) - Rank: 2, Acc: 0.5207
✅ Loaded RandomForest_X (pkl) - Rank: 3, Acc: 0.5199
✅ Loaded RandomForest_Full (pkl) - Rank: 4, Acc: 0.5195
✅ Loaded GRU_Full (keras) - Rank: 5, Acc: 0.5186
✅ Loaded LSTM_X (keras) - Rank: 6, Acc: 0.5179
✅ Loaded LightGBM_X (pkl) - Rank: 7, Acc: 0.5167
✅ Loaded LSTM_Attention_Full (keras) - Rank: 8, Acc: 0.5161
✅ Loaded CatBoost_Full (pkl) - Rank: 9, Acc: 0.5159
✅ Loaded CatBoost_X (pkl) - Rank: 10, Acc: 0.5157
✅ Loaded LSTM_Full (keras) - Rank: 11, Acc: 0.5157
✅ Loaded XGBoost_Full (pkl) - Rank: 12, Acc: 0.5150
✅ Loaded LSTM_Attention_X (keras) - Rank: 13, Acc: 0.5150
✅ Loaded NN_Full (keras) - Rank: 14, Acc: 0.5145
✅ Loaded XGBoost_X (pkl) - Rank: 15, Acc: 0.5143
✅ Loaded GRU_X (keras) - Rank: 16, Acc: 0.5140
✅ Loaded CNN_Full (keras) - Rank: 17, A

,Model,Type,Class,Rank,Train_Acc,Forward_Acc,Feature_Set
0,AdaBoost_Full,pkl,AdaBoostClassifier,1,0.5358,0.5212,Full
1,AdaBoost_X,pkl,AdaBoostClassifier,2,0.5359,0.5207,X
2,RandomForest_X,pkl,RandomForestClassifier,3,0.5351,0.5199,X
3,RandomForest_Full,pkl,RandomForestClassifier,4,0.5354,0.5195,Full
4,GRU_Full,keras,Sequential,5,0.5286,0.5186,Full
5,LSTM_X,keras,Sequential,6,0.5341,0.5179,X
6,LightGBM_X,pkl,LGBMClassifier,7,0.5317,0.5167,X
7,LSTM_Attention_Full,keras,Functional,8,0.5309,0.5161,Full
8,CatBoost_Full,pkl,CatBoostClassifier,9,0.5349,0.5159,Full
9,CatBoost_X,pkl,CatBoostClassifier,10,0.5347,0.5157,X


In [ ]:
# ============================================================================
# LOAD SCALER VALUES FROM CONFIG
# ============================================================================
import os
import json

# Load scaler values from config
RESULTS_ROOT = r"C:\Users\aram\OneDrive\Bureau\ding\testing a strategy\python_classes\results"
RESULTS_5m_FOLDER = None
for folder_name in os.listdir(RESULTS_ROOT):
    if folder_name.startswith('ETH_USD_5m_') and os.path.isdir(os.path.join(RESULTS_ROOT, folder_name)):
        RESULTS_5m_FOLDER = os.path.join(RESULTS_ROOT, folder_name)
        break

if RESULTS_5m_FOLDER:
    with open(os.path.join(RESULTS_5m_FOLDER, "config.json"), "r") as f:
        config_5m = json.load(f)
    
    scaler_5m = config_5m["scaler"]
    print(f"Config loaded from: {os.path.basename(RESULTS_5m_FOLDER)}")
    print(f"Scaler keys: {list(scaler_5m.keys())}")
    
    x_mean_dict_5m = scaler_5m["X_mean"]
    x_std_dict_5m = scaler_5m["X_std"]
    x_full_mean_dict_5m = scaler_5m["X_full_mean"]
    x_full_std_dict_5m = scaler_5m["X_full_std"]
else:
    print("ERROR: No 5m results folder found!")

In [6]:
# ============================================================================

# LOAD 5M DATA FROM YFINANCE

# ============================================================================

import yfinance as yf

import pandas as pd

import numpy as np

import warnings

warnings.filterwarnings('ignore')



SYMBOL_5M = 'ETH-USD'  # Change if needed



print("🚀 Fetching 5m data for:", SYMBOL_5M)



try:

    ticker = yf.Ticker(SYMBOL_5M)



    # Primary: max period (note: 5m data limited to 60 days max)

    df_5m = ticker.history(period='60d', interval='5m', prepost=True, actions=False)



    # Fallbacks if empty

    if df_5m is None or df_5m.empty:

        print("⚠️ primary attempt empty — trying period='30d'...")

        df_5m = ticker.history(period='30d', interval='5m', prepost=True, actions=False)



    if df_5m is None or df_5m.empty:

        print("⚠️ still empty — trying yf.download(period='7d')...")

        df_5m = yf.download(SYMBOL_5M, period='7d', interval='5m', progress=False)



    if df_5m is None or df_5m.empty:

        print("❌ No 5m data retrieved from yfinance.")

        DF_5M_MAX = pd.DataFrame()

    else:

        # Clean

        df_5m = df_5m[~df_5m.index.duplicated(keep='first')]

        df_5m = df_5m.sort_index()

        df_5m = df_5m.dropna(how='all')

        df_5m = df_5m.ffill()



        # Derived columns

        df_5m['Returns'] = df_5m['Close'].pct_change()

        df_5m['Log_Return'] = np.log(df_5m['Close'] / df_5m['Close'].shift(1))



        # Store

        DF_5M_MAX = df_5m.copy()



        print("✅ Data stored in variable: DF_5M_MAX")

        print(f"   Shape: {DF_5M_MAX.shape}")

        print(f"   Date range: {DF_5M_MAX.index[0]} to {DF_5M_MAX.index[-1]}")



        print("\n🔍 FIRST 3 ROWS:")

        display(DF_5M_MAX.head(3))

        print("\n🔍 LAST 3 ROWS:")

        display(DF_5M_MAX.tail(3))



except Exception as e:

    print(f"❌ Error fetching data: {e}")

    import traceback

    traceback.print_exc()

    DF_5M_MAX = pd.DataFrame()


🚀 Fetching 5m data for: ETH-USD
✅ Data stored in variable: DF_5M_MAX
   Shape: (17154, 7)
   Date range: 2025-12-19 00:00:00+00:00 to 2026-02-16 13:28:00+00:00

🔍 FIRST 3 ROWS:


,Open,High,Low,Close,Volume,Returns,Log_Return
Datetime,,,,,,,
2025-12-19 00:00:00+00:00,2826.969727,2836.665283,2826.203369,2833.876465,0,NaN,NaN
2025-12-19 00:05:00+00:00,2826.888916,2829.092041,2825.734131,2829.092041,0,-0.001688,-0.001690
2025-12-19 00:10:00+00:00,2829.862793,2831.097900,2828.728516,2828.728516,679553024,-0.000128,-0.000129



🔍 LAST 3 ROWS:


,Open,High,Low,Close,Volume,Returns,Log_Return
Datetime,,,,,,,
2026-02-16 13:20:00+00:00,2009.609131,2009.609131,1999.256104,2001.915527,8650752,-0.004752,-0.004764
2026-02-16 13:25:00+00:00,1999.699951,1999.699951,1989.572021,1989.572021,21243904,-0.006166,-0.006185
2026-02-16 13:28:00+00:00,1987.795776,1987.795776,1987.795776,1987.795776,0,-0.000893,-0.000893


In [7]:
def calculate_indicators_5m(df):
    """
    Calculate technical indicators for 5m feature set.
    """
    df = df.copy()

    # Target variable (for reference)
    df["return"] = df["Close"].pct_change()
    df["target"] = (df["return"].shift(-1) > 0).astype(int)

    # Trend Indicators
    df["EMA_20"] = tal.EMA(df["Close"], timeperiod=20)
    df["SAR"] = tal.SAR(df['High'], df['Low'], acceleration=0.02, maximum=0.2)
    df["SMA_10"] = df["Close"].rolling(10).mean()
    df["SMA_20"] = df["Close"].rolling(20).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()
    df["STD_10"] = df["Close"].rolling(10).std()
    df["STD_20"] = df["Close"].rolling(20).std()

    # Momentum Indicators
    df["RSI_20"] = tal.RSI(df["Close"], timeperiod=20)
    df["ROC_10"] = df["Close"].pct_change(10)
    df["CCI_20"] = tal.CCI(df['High'], df['Low'], df['Close'], timeperiod=20)
    df["MACD"], df["MACD_signal"], df["MACD_hist"] = tal.MACD(df['Close'])

    # Volatility Indicators
    df["ATR_14"] = tal.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)
    sma = df["Close"].rolling(20).mean()
    std = df["Close"].rolling(20).std()
    df["BB_upper"] = sma + (std * 2)
    df["BB_lower"] = sma - (std * 2)
    df["BW_20"] = (df["BB_upper"] - df["BB_lower"]) / sma

    # Chaikin Money Flow (CMF)
    def calculate_cmf(high, low, close, volume, period=20):
        """Calculate Chaikin Money Flow"""
        mfv = ((close - low) - (high - close)) / (high - low + 1e-10) * volume
        cmf = mfv.rolling(period).sum() / volume.rolling(period).sum()
        return cmf
    
    df["CMF_20"] = calculate_cmf(df['High'], df['Low'], df['Close'], df['Volume'], period=20)

    # Volume lags
    df["Volume_lag1"] = df["Volume"].shift(1)
    df["Volume_lag2"] = df["Volume"].shift(2)

    # Price-based Indicators
    df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))

    # Lag features
    for i in range(1, 6):
        df[f"Lag_{i}"] = df["return"].shift(i)

    # Keep only required 5m feature columns + target/return
    required_cols = [
        "Open",
        "Close",
        "Volume",
        "EMA_20",
        "SAR",
        "RSI_20",
        "ROC_10",
        "CCI_20",
        "MACD",
        "MACD_signal",
        "MACD_hist",
        "ATR_14",
        "BB_lower",
        "BW_20",
        "SMA_10",
        "SMA_20",
        "SMA_50",
        "STD_10",
        "STD_20",
        "CMF_20",
        "Log_Return",
        "Lag_1",
        "Lag_2",
        "Lag_3",
        "Lag_4",
        "Lag_5",
        "Volume_lag1",
        "Volume_lag2",
    ]

    # Drop rows with NaNs in required columns
    df = df.dropna(subset=required_cols + ["target", "return"])

    return df[required_cols + ["target", "return"]]


def _assert_features_match(tf, feature_cols, selected_features, full_features):
    selected_set = set(selected_features or [])
    full_set = set(full_features or [])
    extracted_set = set(feature_cols or [])

    if selected_set and not selected_set.issubset(extracted_set):
        missing = sorted(selected_set - extracted_set)
        raise ValueError(
            f"Feature mismatch for {tf} (selected_features). "
            f"Missing: {missing}"
        )

    if full_set and selected_set and not selected_set.issubset(full_set):
        missing_from_full = sorted(selected_set - full_set)
        raise ValueError(
            f"Feature mismatch for {tf} (full_features). "
            f"Selected not in full_features: {missing_from_full}"
        )


# Define feature sets for 5m (from your config)
X_SELECTED_FEATURES_5m = [
    "Open",
    "Volume",
    "RSI_20",
    "ROC_10",
    "CCI_20",
    "MACD",
    "MACD_hist",
    "ATR_14",
    "BW_20",
    "CMF_20",
    "Log_Return",
    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_4",
    "Lag_5",
    "Volume_lag1",
    "Volume_lag2",
]

X_FULL_FEATURES_5m = [
    "Open",
    "Close",
    "Volume",
    "EMA_20",
    "SAR",
    "RSI_20",
    "ROC_10",
    "CCI_20",
    "MACD",
    "MACD_signal",
    "MACD_hist",
    "ATR_14",
    "BB_lower",
    "BW_20",
    "SMA_10",
    "SMA_20",
    "SMA_50",
    "STD_10",
    "STD_20",
    "CMF_20",
    "Log_Return",
    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_4",
    "Lag_5",
    "Volume_lag1",
    "Volume_lag2",
]

# Prepare features for 5m timeframe using already-loaded 5m data
processed_by_timeframe = {}
feature_cols_by_timeframe = {}
X_by_timeframe = {}

if 'DF_5M_MAX' in globals() and isinstance(DF_5M_MAX, pd.DataFrame) and not DF_5M_MAX.empty:
    df_5m_source = DF_5M_MAX.copy()
    print("Using DF_5M_MAX as source for 5m features")
else:
    raise ValueError("No loaded 5m data found. Please run the 5m yfinance loader cell first.")

tf = "5m"

df_proc = calculate_indicators_5m(df_5m_source)
processed_by_timeframe[tf] = df_proc

selected_feats = X_SELECTED_FEATURES_5m if 'X_SELECTED_FEATURES_5m' in globals() else []
full_feats = X_FULL_FEATURES_5m if 'X_FULL_FEATURES_5m' in globals() else []

# Use full feature set instead of selected features
feature_cols_tf = full_feats if full_feats else [
    col for col in df_proc.columns if col not in ["target", "return"]
]

_assert_features_match(
    tf,
    df_proc.columns.tolist(),
    selected_feats,
    full_feats
)

X_tf = df_proc[feature_cols_tf].copy()

feature_cols_by_timeframe[tf] = feature_cols_tf
X_by_timeframe[tf] = X_tf

tf_safe = tf.replace("-", "_").replace(".", "_")
globals()[f"DF_PROCESSED_{tf_safe}"] = df_proc
globals()[f"FEATURE_COLS_{tf_safe}"] = feature_cols_tf
globals()[f"X_{tf_safe}"] = X_tf

print(f"✅ {tf}: processed={df_proc.shape} | features={len(feature_cols_tf)} | X={X_tf.shape}")
print(f"\n📋 Feature Configuration:")
print(f"  - X_selected_features: {len(X_SELECTED_FEATURES_5m)} features")
print(f"  - X_full_features: {len(X_FULL_FEATURES_5m)} features")

df_processed = df_proc
feature_cols = feature_cols_tf
X = X_tf
TIMEFRAME = tf
print(f"\n✅ Active timeframe ready: {TIMEFRAME}")

Using DF_5M_MAX as source for 5m features
✅ 5m: processed=(16995, 30) | features=28 | X=(16995, 28)

📋 Feature Configuration:
  - X_selected_features: 18 features
  - X_full_features: 28 features

✅ Active timeframe ready: 5m


In [ ]:
# ============================================================================
# BUILD ENSEMBLE PREDICTIONS FROM TOP 3 MODELS
# ============================================================================
import pandas as pd
import numpy as np

print("="*70)
print("🚀 BUILDING ENSEMBLE PREDICTIONS FROM TOP 3 MODELS")
print("="*70)

# Select top 3 models by forward_acc
if len(MODELS_5m) == 0:
    raise ValueError("No models loaded. Run model loading cell first.")

top_3_models = sorted(
    [(name, info) for name, info in MODELS_5m.items()],
    key=lambda x: x[1]['forward_acc'],
    reverse=True
)[:3]

print(f"\n📊 Top 3 models selected by forward accuracy:")
for i, (name, info) in enumerate(top_3_models, 1):
    print(f"   {i}. {name} - Acc: {info['forward_acc']:.4f}, Features: {info['feature_set']}")

# Prepare features (full set, 28 features)
full_features_list = X_FULL_FEATURES_5m if 'X_FULL_FEATURES_5m' in globals() else []
selected_features_list = X_SELECTED_FEATURES_5m if 'X_SELECTED_FEATURES_5m' in globals() else []

# Create feature matrices
if not DF_PROCESSED_5m.empty:
    X_full = DF_PROCESSED_5m[full_features_list].copy()
    X_selected = DF_PROCESSED_5m[selected_features_list].copy()
    
    # Drop NaNs separately for each feature set
    X_full = X_full.dropna()
    X_selected = X_selected.dropna()
    
    # Find common index across both
    common_index = X_full.index.intersection(X_selected.index)
    
    X_full = X_full.loc[common_index]
    X_selected = X_selected.loc[common_index]
    
    print(f"\n📋 Feature matrices created:")
    print(f"   - X_full (28 features): {X_full.shape}")
    print(f"   - X_selected (18 features): {X_selected.shape}")
    print(f"   - Common index: {len(common_index)} rows")
else:
    raise ValueError("DF_PROCESSED_5m is empty. Run indicators cell first.")

# Generate predictions from each model
model_predictions = {}
model_features_used = []
TOP_3_MODEL_NAMES = []
MODEL_FEATURES_USED = []

for model_name, model_info in top_3_models:
    model = model_info['model']
    
    # Determine feature requirement
    try:
        n_features_expected = model.n_features_in_
    except:
        # Try alternative attributes
        try:
            n_features_expected = len(model.feature_names_in_)
        except:
            n_features_expected = None
    
    # Select feature set based on model requirement
    if n_features_expected:
        if n_features_expected == len(selected_features_list):
            X_model = X_selected.copy()
            feature_type = f"selected ({len(selected_features_list)} features)"
        elif n_features_expected == len(full_features_list):
            X_model = X_full.copy()
            feature_type = f"full ({len(full_features_list)} features)"
        else:
            # Default to full features if mismatch
            X_model = X_full.copy()
            feature_type = f"full ({len(full_features_list)} features)"
    else:
        # Default to full features
        X_model = X_full.copy()
        feature_type = f"full ({len(full_features_list)} features)"
    
    # Determine which scaler to use based on feature set
    feature_set = model_info['feature_set']
    if feature_set == 'Full' or n_features_expected == len(full_features_list):
        mean_dict = x_full_mean_dict_5m
        std_dict = x_full_std_dict_5m
    else:
        mean_dict = x_mean_dict_5m
        std_dict = x_std_dict_5m

    # Z-score normalize using config values
    mean_s = pd.Series(mean_dict).reindex(X_model.columns).astype(float)
    std_s = pd.Series(std_dict).reindex(X_model.columns).astype(float).replace(0, np.nan)
    X_z = (X_model - mean_s) / std_s
    X_z = X_z.replace([np.inf, -np.inf], np.nan).dropna()
    
    # Generate predictions for this model
    try:
        # Check if this is an LSTM/Neural Network model
        is_lstm = False
        if KERAS_AVAILABLE and hasattr(model, 'predict'):
            # Check if it's a Keras model by looking for typical Keras attributes
            if hasattr(model, 'input_shape') or str(type(model).__module__).startswith('tensorflow'):
                is_lstm = True
                print(f"\n🔹 {model_name}: Detected as LSTM/Neural Network model")
        
        if is_lstm and KERAS_AVAILABLE:
            try:
                # For LSTM models: reshape to 3D (samples, 1, features) - single timestep
                X_z_reshaped = X_z.values.reshape((X_z.shape[0], 1, X_z.shape[1]))
                print(f"   - Reshaping for LSTM: {X_z.shape} → {X_z_reshaped.shape}")
                
                # Try to build the model with correct input shape if it has unknown rank
                if hasattr(model, 'built') and not model.built:
                    print(f"   - Building LSTM model with input shape: {X_z_reshaped.shape}")
                    model.build(input_shape=(None, X_z_reshaped.shape[1], X_z_reshaped.shape[2]))
                
                # Get predictions with small batch size to avoid memory issues
                proba = model.predict(X_z_reshaped, verbose=0, batch_size=128)
                
                # Handle different output shapes
                if len(proba.shape) > 1 and proba.shape[1] > 1:
                    prob_up_model = proba[:, 1]  # Binary classification: class 1 probability
                else:
                    prob_up_model = proba.flatten()  # Already probability for up class
                
                print(f"   - LSTM prediction successful: {len(prob_up_model)} predictions")
            except Exception as lstm_error:
                print(f"   ⚠️ LSTM prediction failed: {str(lstm_error)}")
                print(f"   - Attempting fallback flatten prediction...")
                try:
                    # Flatten and try 2D prediction
                    X_z_2d = X_z.values
                    proba = model.predict(X_z_2d, verbose=0, batch_size=128)
                    prob_up_model = proba.flatten() if len(proba.shape) > 1 else proba
                except:
                    # Skip this model if both attempts fail
                    print(f"   - Skipping {model_name} due to prediction errors")
                    raise lstm_error
        else:
            # Standard sklearn models
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(X_z)
                prob_up_model = proba[:, 1]
            else:
                # Fallback: use decision function or predictions
                if hasattr(model, 'decision_function'):
                    scores = model.decision_function(X_z)
                    prob_up_model = 1 / (1 + np.exp(-scores))
                else:
                    preds = model.predict(X_z)
                    prob_up_model = preds.astype(float)
        
        # Clip to valid range
        prob_up_model = np.clip(prob_up_model, 0, 1)
        
        # Store as pandas Series with index for proper alignment
        model_predictions[model_name] = pd.Series(prob_up_model, index=X_model.index)
        model_features_used.append(feature_type)
        TOP_3_MODEL_NAMES.append(model_name)
        MODEL_FEATURES_USED.append(feature_type)
        
        print(f"✅ {model_name}: {feature_type} | predictions: {len(prob_up_model)} | mean prob: {prob_up_model.mean():.4f}")
        
    except Exception as e:
        print(f"❌ {model_name}: Prediction failed - {str(e)}")
        print(f"   - Model type: {type(model)}")
        print(f"   - Input shape: {X_z.shape}")
        import traceback
        traceback.print_exc()
        continue

# Align predictions across models
if len(model_predictions) == 0:
    raise ValueError("No successful model predictions generated!")

# Find common index across all predictions
common_pred_index = list(model_predictions.values())[0].index
for pred_series in list(model_predictions.values())[1:]:
    common_pred_index = common_pred_index.intersection(pred_series.index)

print(f"\n🔀 Aligning predictions:")
print(f"   - Common prediction index: {len(common_pred_index)} timestamps")

# Create aligned DataFrame for ensemble averaging
aligned_predictions = pd.DataFrame()
for model_name, pred_series in model_predictions.items():
    aligned_predictions[model_name] = pred_series.loc[common_pred_index]

# Calculate ensemble as mean of aligned predictions
prob_up_ensemble = aligned_predictions.mean(axis=1).values

print(f"   - Ensemble probabilities: {len(prob_up_ensemble)} values")

# Create output dataframe with predictions
PREDICTIONS_5M_DF = DF_PROCESSED_5m.loc[common_pred_index].copy()
PREDICTIONS_5M_DF['Pred_Prob_Up'] = prob_up_ensemble
PREDICTIONS_5M_DF['True_Target'] = PREDICTIONS_5M_DF['target']

# Add individual model predictions
for i, (model_name, pred_series) in enumerate(model_predictions.items(), 1):
    PREDICTIONS_5M_DF[f'Prob_Model_{i}'] = pred_series.loc[common_pred_index].values

print(f"\n✅ PREDICTIONS_5M_DF created: {PREDICTIONS_5M_DF.shape}")
print(f"\n📊 Ensemble Prediction Statistics:")
print(f"   - Ensemble prob range: [{prob_up_ensemble.min():.4f}, {prob_up_ensemble.max():.4f}]")
print(f"   - Mean ensemble prob: {prob_up_ensemble.mean():.4f}")
print(f"   - Std ensemble prob: {prob_up_ensemble.std():.4f}")

print("\n" + "="*70)

🚀 BUILDING ENSEMBLE PREDICTIONS FROM TOP 3 MODELS

📊 Top 3 models selected by forward accuracy:
   1. AdaBoost_Full - Acc: 0.5212, Features: Full
   2. AdaBoost_X - Acc: 0.5207, Features: X
   3. RandomForest_X - Acc: 0.5199, Features: X

📋 Feature matrices created:
   - X_full (28 features): (16995, 28)
   - X_selected (18 features): (16995, 18)
   - Common index: 16995 rows
✅ AdaBoost_Full: full (28 features) | predictions: 16995 | mean prob: 0.4849
✅ AdaBoost_X: selected (18 features) | predictions: 16995 | mean prob: 0.4884
✅ RandomForest_X: selected (18 features) | predictions: 16995 | mean prob: 0.4970

🔀 Aligning predictions:
   - Common prediction index: 16995 timestamps
   - Ensemble probabilities: 16995 values

✅ PREDICTIONS_5M_DF created: (16995, 35)

📊 Ensemble Prediction Statistics:
   - Ensemble prob range: [0.3102, 0.7130]
   - Mean ensemble prob: 0.4901
   - Std ensemble prob: 0.0879



In [9]:
# ============================================================================
# Summary Statistics for Predicted Probabilities
# ============================================================================

# Check if the predictions dataframe exists
if 'PREDICTIONS_5M_DF' in globals() and isinstance(PREDICTIONS_5M_DF, pd.DataFrame) and not PREDICTIONS_5M_DF.empty:
    # Extract the probabilities
    prob_up = PREDICTIONS_5M_DF['Pred_Prob_Up']
    
    # Calculate the range of probabilities
    prob_range = (prob_up.min(), prob_up.max())
    
    # Calculate the number of rows
    num_rows = prob_up.shape[0]
    
    # Calculate the mean and standard deviation
    prob_mean = prob_up.mean()
    prob_std = prob_up.std()
    
    # Display the summary statistics
    print("📊 Summary Statistics for Predicted Probabilities:")
    print(f"  - Range: {prob_range}")
    print(f"  - Number of Rows: {num_rows}")
    print(f"  - Mean Probability: {prob_mean:.4f}")
    print(f"  - Standard Deviation: {prob_std:.4f}")
else:
    print("❌ 'PREDICTIONS_5M_DF' is not available or is empty.")

📊 Summary Statistics for Predicted Probabilities:
  - Range: (np.float64(0.31017552791233133), np.float64(0.7129723084301088))
  - Number of Rows: 16995
  - Mean Probability: 0.4901
  - Standard Deviation: 0.0879


In [ ]:
# ============================================================================
# STOP LOSS STRATEGY FUNCTION WITH PROPER UNIT TRACKING
# ============================================================================

def run_strategy_with_stop_loss(df, buy_threshold=0.6, sell_threshold=0.2, 
                                  stop_loss_pct=0.01, initial_balance=1000, 
                                  trading_cost=0.001):
    """
    Run a trading strategy with stop loss that properly tracks units.
    
    Args:
        df: DataFrame with 'Close', 'Low', 'prob_up' columns
        buy_threshold: Probability threshold to enter position
        sell_threshold: Probability threshold to exit position
        stop_loss_pct: Stop loss percentage (e.g., 0.01 = 1%)
        initial_balance: Starting balance in USD
        trading_cost: Trading cost as fraction (e.g., 0.001 = 0.1%)
    
    Returns:
        final_balance: Final balance after all trades
        trades: List of trade dictionaries
    """
    balance = initial_balance
    units = 0  # Number of units owned - CRITICAL FIX
    position = 0
    entry_price = 0
    trades = []
    
    for i in range(len(df)):
        row = df.iloc[i]
        current_price = row['Close']
        prob = row['prob_up']
        low_price = row['Low']
        
        if position == 1:
            stop_price = entry_price * (1 - stop_loss_pct)
            
            if low_price <= stop_price:
                exit_price = stop_price
                proceeds = units * exit_price * (1 - trading_cost)  # FIXED
                trades.append({
                    'type': 'STOP_LOSS',
                    'entry': entry_price,
                    'exit': exit_price,
                    'return': (exit_price / entry_price) - 1,
                    'balance': proceeds
                })
                balance = proceeds
                position = 0
                entry_price = 0
                units = 0
                continue
            
            elif prob < sell_threshold:
                exit_price = current_price
                proceeds = units * exit_price * (1 - trading_cost)  # FIXED
                trades.append({
                    'type': 'SELL',
                    'entry': entry_price,
                    'exit': exit_price,
                    'return': (exit_price / entry_price) - 1,
                    'balance': proceeds
                })
                balance = proceeds
                position = 0
                entry_price = 0
                units = 0
        
        elif position == 0 and prob > buy_threshold:
            entry_price = current_price * (1 + trading_cost)
            units = balance / entry_price  # FIXED
            position = 1
            balance = 0
    
    if position == 1:
        exit_price = df.iloc[-1]['Close']
        balance = units * exit_price * (1 - trading_cost)  # FIXED
    
    return balance, trades

print("✅ run_strategy_with_stop_loss function defined")

In [10]:
# ============================================================================
# Simple Buy/Neutral Strategy Based on Predicted Probabilities with Buy and Sell Thresholds
# ============================================================================

# Define thresholds for the probability to trigger buy and sell actions
buy_threshold = 0.66  # Buy when probability exceeds this threshold
sell_threshold = 0.304 # Sell when probability goes below this threshold
stop_loss_pct = 0.025  # 1.5% stop loss
trading_cost_pct = 0.001  # 0.1% trading cost

# Initialize strategy variables
initial_balance = 1000.0  # Starting balance in USD
balance = initial_balance  # Current balance
position = 0  # 0 means no position, 1 means holding position
entry_price = 0  # Price when bought (including trading costs)
trade_history = []  # To store trade history for performance evaluation

# Check if the predictions dataframe exists
if 'PREDICTIONS_5M_DF' in globals() and isinstance(PREDICTIONS_5M_DF, pd.DataFrame) and not PREDICTIONS_5M_DF.empty:
    # Iterate through each row to simulate the strategy
    for _, row in PREDICTIONS_5M_DF.iterrows():
        price = row['Close']
        prob = row['Pred_Prob_Up']

        # If the probability exceeds the buy threshold and we are not already holding a position
        if prob > buy_threshold and position == 0:
            # Execute buy: enter position (apply trading cost)
            entry_price = price * (1 + trading_cost_pct)
            position = 1
            trade_history.append({
                'action': 'BUY',
                'price': price,
                'entry_price': entry_price,
                'prob': prob,
                'balance_before': balance
            })

        # If we are holding a position, check stop loss or sell threshold
        elif position == 1:
            stop_loss_price = entry_price * (1 - stop_loss_pct)

            # PRIORITY 1: Check stop loss first
            if price <= stop_loss_price:
                # Stop loss triggered (apply trading cost on exit)
                exit_price = price * (1 - trading_cost_pct)
                pct_return = (exit_price / entry_price) - 1
                balance *= (exit_price / entry_price)
                
                trade_history.append({
                    'action': 'STOP LOSS',
                    'price': price,
                    'exit_price': exit_price,
                    'entry_price': entry_price,
                    'prob': prob,
                    'return_pct': pct_return * 100,
                    'balance_after': balance
                })
                position = 0

            # PRIORITY 2: Check sell threshold
            elif prob < sell_threshold:
                # Execute sell: exit position (apply trading cost on exit)
                exit_price = price * (1 - trading_cost_pct)
                pct_return = (exit_price / entry_price) - 1
                balance *= (exit_price / entry_price)
                
                trade_history.append({
                    'action': 'SELL',
                    'price': price,
                    'exit_price': exit_price,
                    'entry_price': entry_price,
                    'prob': prob,
                    'return_pct': pct_return * 100,
                    'balance_after': balance
                })
                position = 0

    # Close any remaining open position at last price
    if position == 1:
        last_price = PREDICTIONS_5M_DF['Close'].iloc[-1]
        exit_price = last_price * (1 - trading_cost_pct)
        pct_return = (exit_price / entry_price) - 1
        balance *= (exit_price / entry_price)
        
        trade_history.append({
            'action': 'CLOSE (EOD)',
            'price': last_price,
            'exit_price': exit_price,
            'entry_price': entry_price,
            'prob': PREDICTIONS_5M_DF['Pred_Prob_Up'].iloc[-1],
            'return_pct': pct_return * 100,
            'balance_after': balance
        })

    # Final report
    final_balance = balance
    total_return = (final_balance / initial_balance - 1) * 100
    
    # Count trades by type
    buy_count = sum(1 for t in trade_history if t['action'] == 'BUY')
    sell_count = sum(1 for t in trade_history if t['action'] == 'SELL')
    stop_loss_count = sum(1 for t in trade_history if t['action'] == 'STOP LOSS')
    
    # Calculate win rate from exits
    exit_trades = [t for t in trade_history if 'return_pct' in t]
    winning_trades = [t for t in exit_trades if t['return_pct'] > 0]
    win_rate = (len(winning_trades) / len(exit_trades) * 100) if exit_trades else 0
    
    print(f"\n{'='*60}")
    print(f"✅ STRATEGY EXECUTION COMPLETE")
    print(f"{'='*60}")
    print(f"\n💰 FINAL RESULTS:")
    print(f"  - Initial Balance:  ${initial_balance:.2f}")
    print(f"  - Final Balance:    ${final_balance:.2f}")
    print(f"  - Total Return:     {total_return:+.2f}%")
    print(f"  - Profit/Loss:      ${final_balance - initial_balance:+.2f}")
    
    print(f"\n📊 TRADING STATISTICS:")
    print(f"  - Total Entries:    {buy_count}")
    print(f"  - Regular Exits:    {sell_count}")
    print(f"  - Stop Loss Exits:  {stop_loss_count}")
    print(f"  - Win Rate:         {win_rate:.2f}%")
    
    print(f"\n📋 TRADE HISTORY (Last 10 trades):")
    for i, trade in enumerate(trade_history[-10:], 1):
        if trade['action'] == 'BUY':
            print(f"  {i}. {trade['action']} @ ${trade['price']:.4f} (prob: {trade['prob']:.4f}) | Entry: ${trade['entry_price']:.4f}")
        else:
            print(f"  {i}. {trade['action']} @ ${trade['price']:.4f} (prob: {trade['prob']:.4f}) | Return: {trade['return_pct']:+.2f}% | Balance: ${trade['balance_after']:.2f}")
    
    # ============================================================================
    # PERFORMANCE METRICS - Sharpe Ratio and Maximum Drawdown
    # ============================================================================
    
    # Extract strategy returns
    strategy_returns = [trade['return_pct'] / 100 for trade in trade_history if 'return_pct' in trade]

    # Convert to pandas series for easier calculations
    strategy_returns = pd.Series(strategy_returns)
    
    # Sharpe Ratio (Risk-Free Rate assumed to be 0)
    excess_return = strategy_returns.mean()  # Excess return = mean of strategy returns
    volatility = strategy_returns.std()  # Volatility (standard deviation of returns)
    sharpe_ratio = excess_return / volatility if volatility != 0 else 0
    
    # Maximum Drawdown (MDD)
    def max_drawdown(equity_curve):
        peak = equity_curve.expanding(min_periods=1).max()
        drawdown = (equity_curve - peak) / peak
        return drawdown.min() * 100
    
    # Calculate cumulative returns for MDD calculation
    equity_curve = (1 + strategy_returns).cumprod()
    mdd = max_drawdown(equity_curve)
    
    # Print performance metrics
    print(f"\n📊 PERFORMANCE METRICS:")
    print(f"  - Sharpe Ratio:     {sharpe_ratio:.2f}")
    print(f"  - Maximum Drawdown: {mdd:.2f}%")

else:
    print("❌ 'PREDICTIONS_5M_DF' is not available or is empty.")


✅ STRATEGY EXECUTION COMPLETE

💰 FINAL RESULTS:
  - Initial Balance:  $1000.00
  - Final Balance:    $577.18
  - Total Return:     -42.28%
  - Profit/Loss:      $-422.82

📊 TRADING STATISTICS:
  - Total Entries:    20
  - Regular Exits:    0
  - Stop Loss Exits:  19
  - Win Rate:         5.00%

📋 TRADE HISTORY (Last 10 trades):
  1. BUY @ $2121.4846 (prob: 0.6722) | Entry: $2123.6061
  2. STOP LOSS @ $2067.8074 (prob: 0.5793) | Return: -2.72% | Balance: $613.61
  3. BUY @ $2023.7184 (prob: 0.6882) | Entry: $2025.7421
  4. STOP LOSS @ $1952.8756 (prob: 0.6648) | Return: -3.69% | Balance: $590.94
  5. BUY @ $1941.4506 (prob: 0.6909) | Entry: $1943.3920
  6. STOP LOSS @ $1891.3473 (prob: 0.6275) | Return: -2.78% | Balance: $574.54
  7. BUY @ $1857.2977 (prob: 0.6610) | Entry: $1859.1550
  8. STOP LOSS @ $1785.1901 (prob: 0.5721) | Return: -4.07% | Balance: $551.13
  9. BUY @ $1894.2823 (prob: 0.6702) | Entry: $1896.1766
  10. CLOSE (EOD) @ $1987.7958 (prob: 0.6113) | Return: +4.73% | Bal

In [11]:
# ============================================================================
# OPTIMIZE BUY/SELL THRESHOLDS AND STOP LOSS WITH GRID SEARCH
# ============================================================================
import numpy as np
import pandas as pd

if 'PREDICTIONS_5M_DF' not in globals() or PREDICTIONS_5M_DF.empty:
    raise ValueError("PREDICTIONS_5M_DF is missing. Run the predictions dataframe cell first.")

# Configuration
initial_balance = 1000.0
trading_cost = 0.001  # 0.1%

# Threshold and stop loss grids
buy_grid = np.round(np.linspace(0.50, 0.80, 16), 3)
sell_grid = np.round(np.linspace(0.20, 0.49, 15), 3)
stop_loss_grid = np.round(np.linspace(0.01, 0.05, 9), 4)  # 1% to 5% stop loss

print("📊 OPTIMIZATION PARAMETERS:")
print(f"  - Buy thresholds: {len(buy_grid)} values ({buy_grid[0]:.3f} to {buy_grid[-1]:.3f})")
print(f"  - Sell thresholds: {len(sell_grid)} values ({sell_grid[0]:.3f} to {sell_grid[-1]:.3f})")
print(f"  - Stop loss: {len(stop_loss_grid)} values ({stop_loss_grid[0]:.4f} to {stop_loss_grid[-1]:.4f})")
print(f"  - Total combinations: {len(buy_grid) * len(sell_grid) * len(stop_loss_grid)}")

results = []

for buy_th in buy_grid:
    for sell_th in sell_grid:
        if sell_th >= buy_th:
            continue
        
        for sl_pct in stop_loss_grid:
            balance = initial_balance
            position = 0
            entry_price = None

            for _, row in PREDICTIONS_5M_DF.iterrows():
                prob = row['Pred_Prob_Up']
                price = row['Close']

                if position == 0 and prob >= buy_th:
                    # Buy
                    entry_price = price * (1 + trading_cost)
                    position = 1
                elif position == 1:
                    # Check stop loss first (priority 1)
                    stop_loss_price = entry_price * (1 - sl_pct)
                    if price <= stop_loss_price:
                        # Stop loss triggered
                        exit_price = price * (1 - trading_cost)
                        balance *= (exit_price / entry_price)
                        position = 0
                    # Check sell threshold (priority 2)
                    elif prob <= sell_th:
                        # Sell
                        exit_price = price * (1 - trading_cost)
                        balance *= (exit_price / entry_price)
                        position = 0

            # If still in position, close at last price
            if position == 1:
                last_price = PREDICTIONS_5M_DF['Close'].iloc[-1] * (1 - trading_cost)
                balance *= (last_price / entry_price)

            total_return = (balance / initial_balance) - 1
            results.append({
                'buy_threshold': float(buy_th),
                'sell_threshold': float(sell_th),
                'stop_loss_pct': float(sl_pct),
                'final_balance': float(balance),
                'total_return': float(total_return)
            })

results_df = pd.DataFrame(results).sort_values('total_return', ascending=False)

best_row = results_df.iloc[0]
OPTIMAL_BUY_THRESHOLD = best_row['buy_threshold']
OPTIMAL_SELL_THRESHOLD = best_row['sell_threshold']
OPTIMAL_STOP_LOSS = best_row['stop_loss_pct']

print("\n" + "="*70)
print("✅ OPTIMIZATION COMPLETE")
print("="*70)
print(f"\n🎯 OPTIMAL PARAMETERS (Maximizing Total Return):")
print(f"  - Buy Threshold:   {OPTIMAL_BUY_THRESHOLD:.4f}")
print(f"  - Sell Threshold:  {OPTIMAL_SELL_THRESHOLD:.4f}")
print(f"  - Stop Loss:       {OPTIMAL_STOP_LOSS:.4f} ({OPTIMAL_STOP_LOSS*100:.2f}%)")
print(f"  - Final Balance:   ${best_row['final_balance']:.2f}")
print(f"  - Total Return:    {best_row['total_return']*100:+.2f}%")

print(f"\n📊 TOP 10 PARAMETER COMBINATIONS:")
display(results_df[['buy_threshold', 'sell_threshold', 'stop_loss_pct', 'final_balance', 'total_return']].head(10))

# Create visualization: Heatmaps for best stop loss configurations
print(f"\n📈 ANALYSIS BY STOP LOSS LEVEL:")
for sl in sorted(results_df['stop_loss_pct'].unique())[:3]:  # Show top 3 stop loss levels
    subset = results_df[results_df['stop_loss_pct'] == sl].nlargest(1, 'total_return')
    if not subset.empty:
        row = subset.iloc[0]
        print(f"  - SL {sl*100:.2f}%: Buy={row['buy_threshold']:.3f}, Sell={row['sell_threshold']:.3f}, Return={row['total_return']*100:+.2f}%")

📊 OPTIMIZATION PARAMETERS:
  - Buy thresholds: 16 values (0.500 to 0.800)
  - Sell thresholds: 15 values (0.200 to 0.490)
  - Stop loss: 9 values (0.0100 to 0.0500)
  - Total combinations: 2160

✅ OPTIMIZATION COMPLETE

🎯 OPTIMAL PARAMETERS (Maximizing Total Return):
  - Buy Threshold:   0.7000
  - Sell Threshold:  0.3240
  - Stop Loss:       0.0300 (3.00%)
  - Final Balance:   $1040.30
  - Total Return:    +4.03%

📊 TOP 10 PARAMETER COMBINATIONS:


,buy_threshold,sell_threshold,stop_loss_pct,final_balance,total_return
1408,0.7,0.324,0.030,1040.301182,0.040301
1412,0.7,0.324,0.050,1040.301182,0.040301
1409,0.7,0.324,0.035,1040.301182,0.040301
1407,0.7,0.324,0.025,1040.301182,0.040301
1410,0.7,0.324,0.040,1040.301182,0.040301
1406,0.7,0.324,0.020,1040.301182,0.040301
1411,0.7,0.324,0.045,1040.301182,0.040301
1404,0.7,0.324,0.010,1032.435312,0.032435
1430,0.7,0.366,0.050,1023.835518,0.023836
1428,0.7,0.366,0.040,1023.835518,0.023836



📈 ANALYSIS BY STOP LOSS LEVEL:
  - SL 1.00%: Buy=0.700, Sell=0.324, Return=+3.24%
  - SL 1.50%: Buy=0.700, Sell=0.366, Return=+2.38%
  - SL 2.00%: Buy=0.700, Sell=0.324, Return=+4.03%
